# TP3- Linear Regression

Estimated time: **1.5** hour


In this lab, you'll go over hands-on exercises using Python to further consolidate your understanding about linear regression, in particular the different techniques of regression (simple linear regression, polynomial regression, multiple linear regression). More importantly, you will learn how to fit a regression model to the data using Ordinary Least Squares (OLS) or the LinearRegression method from the scikit-learn library. You will also plot and analyze the residuals. Last but not least, you will learn how to assess the model's performance with the Root Mean Squared Error (RMSE) and R-Squared ($R^2$)

## Objectives


* Get familiar with the dataset
* Sample exercises
  * Simple linear regression, polynomial regression, multiple linear regression
  * Perform regression using either OLS or LinearRegression from scikit-learn
  * Plot and analyze residuals
  * Assess model performance with RMSE and R-Squared
* Assignment questions

----


# Get familiar with the data set

### 📚 Import Libraries


Install required packages for this lab. `pip` will automatically skip already installed packages

In [ ]:
import importlib.util

# import name: pip installation name
required_pkgs = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "statsmodels": "statsmodels",
    "scipy": "scipy",
}

missing_pkgs = [
    pip_name
    for import_name, pip_name in required_pkgs.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_pkgs:
    print("⚠️ Missing packages:", ", ".join(missing_pkgs))
    packages = " ".join(missing_pkgs)
    %pip install $packages
else:
    print("🎉 All packages already installed.")

**Import Libraries**


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import OLSInfluence

# 📝 Example Exercises

# Example 1: Simple Linear Regression

## <span style="color:blue"> Synthesize a data set for simple linear regression </span>

In [ ]:
n = 60
np.random.seed(3)
x = np.append(np.random.uniform(0, 4, n - 2), [2, 2.1])
y = 5.6 * x + np.random.normal(0, 2, n)

# Create a  data frame based on the X and y value
data1 = pd.DataFrame({"x": x, "y": y})

# Plot the data
plt.figure(figsize=(8, 6))
plt.scatter(data1['x'], data1['y'], alpha=0.6)
plt.title(f"Scatterplot of the dataset", fontsize=16)
plt.xlabel("x", fontsize=12)
plt.ylabel("y", fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

## <span style="color:blue">Example 1.1: Obtain the linear regression model with OLS </span>

In [ ]:
# Apply OLS
X = sm.add_constant(data1['x'])  # Add constant term for the intercept
model = sm.OLS(data1['y'], X).fit()

# Extract coefficients
intercept = model.params['const']
slope = model.params['x']

# Print the formula
print(f"Fitted Regression Formula: y = {intercept:.3f} + {slope:.3f} * x")

data1['predicted'] = model.fittedvalues
data1['residuals'] = model.resid

`Ordinary Least Squares`(OLS) is the most widely used methods for fitting a regression model. In the above code, the OLS methods from statsmodels library is used to fit a linear regression model. It **determines the best-fitting line that minimizes the sum of squared residuals between the observed and predicted values**.

Pay attention, the `add_constant()` method adds a constant column with value 1 to the predictors, which is required by OLS to model the intercept.

Based on the **slope** and **intercept** extracted from `model.params`, we can derive the formula corresponding to the fitted regression model. 

Besides, we also store the predicted y-values and residuals in separate columns in the dataframe for later use. Below, it displays the extended dataframe.

In [ ]:
data1.head(10)

## <span style="color:blue">Example 1.2: Visualize the linear regression model </span>

In [ ]:
# Scatter plot of the dataset with the fitted regression line
plt.figure(figsize=(8, 6))

# Scatter plot of observed data
plt.scatter(data1['x'], data1['y'], alpha=0.6, label="Observed Data")

# Regression line
plt.plot(data1['x'], data1['predicted'], color='red', linewidth=2, label="Regression line")

# Titles and labels
plt.title("Regression Model", fontsize=16)
plt.xlabel("Independent Variable (X)", fontsize=12)
plt.ylabel("Dependent Variable (Y)", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

# Show the plot
plt.show()

The above plot visualizes the linear regression model obtained with OLS, using the predicted values. It first generates the scatter plot corresponding to the original data set. And then plot the fitted regression line.

-----

❓ <span style="color:purple;">**Quick test**</span>:  **According to the above plot, can we conclude that the regression model is appropriate?**

- **A.** Yes, because most observations are near the regression line.
- **B.** Yes, because the slope of the regression line is positive.
- **C.** No; we should also inspect the residuals and evaluate RMSE and $R^2$, ideally on unseen data.
- **D.** No; a good regression line must pass through every observation.

<details>
<summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;"><b>C.</b> No; we should also inspect the residuals and evaluate RMSE and $R^2$, ideally on unseen data.</p>

**Explanation**:

<p style="color:blue;">The observations generally lie close to the regression line, suggesting that the model captures much of the relationship between \(X\) and \(Y\). However, this plot alone is not sufficient to conclude that the regression model is appropriate.</p>

<p style="color:blue;">We should also inspect the <b>residual plot</b> and check whether there is any systematic pattern not captured by the model. In addition, metrics such as <b>RMSE</b> and <b>\(R^2\)</b> can be used to assess the model’s performance. Ideally, these metrics should be calculated using <b>unseen test data</b> to evaluate how well the model generalizes to new observations.</p>

</details>

-----

## <span style="color:blue">Example 1.3:  Visualize the residual for each data point. </span>

Residuals are the **differences between the observed values of the dependent variable and the predicted values that the model generates**: residual = actual Y – predicted Y. They represent the **part of the data that the model does not explain**.


In [ ]:
# Scatter plot of the dataset with regression line and residuals
plt.figure(figsize=(10, 7))

# Scatter plot of the observed data
plt.scatter(data1['x'], data1['y'], alpha=0.6, label="Observed data")

# Regression line
plt.plot(data1['x'], data1['predicted'], color='red', linewidth=2, label="Regression line")

# Add vertical lines to visualize residual for each data point
for i in range(len(data1)):
    # The vertical line extends from y to the predicted value, for the i-th data point
    # data.iloc[i] retrieves the i-th data point
    plt.plot([data1.iloc[i]['x'], data1.iloc[i]['x']], [data1.iloc[i]['y'], data1.iloc[i]['predicted']], color='purple', linestyle='--', alpha=0.6)

# Titles and labels
plt.title("Dataset with Fitted Regression Line and Residuals", fontsize=16)
plt.xlabel("Independent Variable (X)", fontsize=12)
plt.ylabel("Dependent Variable (Y)", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

# Show the plot
plt.show()

-----

❓ <span style="color:purple;">**Quick test**</span>: **Which part of the above plot represents the residual for an observation?**

<details>
<summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;"> The vertical dashed purple segment between the observed point and the regression line.</p>

**Explanation**:

<p style="color:blue;">For a given value of \(X\), the regression line provides the predicted value $Y_{\text{predicted}}$, while the blue point represents the observed value $Y_{\text{actual}}$. Their vertical difference is the residual: </p>

<p style="color:blue; text-align:center; font-size:18px;">
  residual = $Y_{\text{actual}} - Y_{\text{predicted}}$
</p>

<p style="color:blue;">An observation above the regression line has a <b>positive residual</b>, whereas an observation below the line has a <b>negative residual</b>. The length of the dashed purple segment represents the magnitude of the residual.</p>

</details>

-----

## <span style="color:blue">Example 1.4:  Plot a scatter plot for the residuals </span>

In [ ]:
# Residual scatter plot
plt.figure(figsize=(8, 6))

# Scatter plot for the residuals
plt.scatter(data1['x'], data1['residuals'], alpha=0.6, label='Residuals')

# Add a horizontal line at 0
plt.axhline(0, color='red', linestyle='--', linewidth=1)

# Titles and labels
plt.title("Residual Plot", fontsize=16)
plt.xlabel("Independent Variable (X)", fontsize=12)
plt.ylabel("Residuals", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

# Show the plot
plt.show()

-----

❓ <span style="color:purple;">**Quick test**</span>: **According to the residual plot above, is the linear regression model found previously a good fit?**

- **A.** Yes; the residuals are randomly scattered around zero, with no visible pattern, which indicates a reasonably good fit.
- **B.** No; a good model should have residuals that are all equal to zero.
- **C.** No; the presence of both positive and negative residuals indicates that the model is incorrect.
- **D.** It is impossible to draw any conclusion from a residual plot.

<details>
<summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;"><b>A.</b> Yes; the residuals are randomly scattered around zero, with no visible pattern, which indicates a reasonably good fit.</p>

**Explanation**:

<p style="color:blue;"> According to the scatter plot, the residuals do not increase or decrease in intensity, and <b>are randomly distributed around zero</b>. There is no obvious pattern in the residuals. This indicates the obtained linear regression model captures reasonably well the relationship between X and Y.</p>

<p style="color:blue;"> The ideal residual would be zero, because that would mean the data point falls exactly on the regression line. And there is no difference between the predicted and observed values. However, this is unlikely to happen. There are going to be some residuals are positive and some residuals are negative, corresponding to some data points falling above the regression line and some data points falling below the regression line. </p>

<p style="color:blue;"> The reason that we want the residuals to be randomly scattered around zero, is because <b>we want the linear model to capture all of (or as much as possible) the pattern in the data, and anything that's left over to be simply random scatter</b>. </p>

</details>

-----

## <span style="color:blue">Example 1.5: Compute manually the Root Mean Squared Error

Root Mean Squared Error (RMSE) is a standard metric used to **evaluate the accuracy of a regression model**. It **quantifies the average magnitude of prediction errors in the same units as the dependent variable (𝑌)**. RMSE is calculated as the square root of the average squared prediction errors (residuals).


$$
\Large
\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n \left( Y_{\text{actual}, i} - Y_{\text{predicted}, i} \right)^2}
$$

OLS doesn't return directly RMSE, we can compute it manually using the above formula. 

In [ ]:
# Sum of Squared Residuals (SSR)
rss = np.sum((data1['y'] - data1['predicted'])**2)

# Number of data points
n = len(data1['y'])

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(rss / n)

print(f"Root Mean Squared Error (RMSE) is: {rmse:.3f}")

## <span style="color:blue">Example 1.6:  Compute manually the R-Sqaured ($R^2$) </span>

R-Squared $R^2$, also called the coefficient of determination, measures the **proportion of variance in the dependent variable (Y) that is explained by the independent variable(s) (X) in the regression model**. 

It provides an indication of how well the regression model fits the data. The value of $R^2$ normally lies between 0 and 1, where:

$R^2=1$: The model explains 100% of the variance in the dependent variable (Y).

$R^2=0$: The model explains none of the variance (i.e., it performs no better than using the mean of y as a prediction).


$$
\Large
R^2 = 1 - \frac{\text{SSR}}{\text{SST}}
$$

$$
\Large
\text{SSR} = \sum_{i=1}^n ( Y_{\text{actual}, i} - Y_{\text{predicted}, i})^2
$$

$$
\Large
\text{SST} = \sum_{i=1}^n (Y_{\text{actual}, i} - \bar{Y})^2
$$

where,
- $Y_{\text{actual}, i}$: actual values.
- $Y_{\text{predicted}, i}$: predicted values from the regression model
- $\bar{Y}$: mean of actual values

$\textbf{SST}$ is the Total Sum of Squares, which represents the total variability in the data.  $\textbf{SSR}$ is the Sum of Squared Residuals, which represents the unexplained variability by the regression model. Hence, $\frac{\textbf{SSR}}{\textbf{SST}}$ represents the proportion of variance that is not explained by the model, and $\textbf{1}-\frac{\textbf{SSR}}{\textbf{SST}}$ represents the proportion of variance that is explained by the model. 

In [ ]:
# Calculate the mean of observed y-values
y_mean = data1['y'].mean()

# Calculate SST (Total Sum of Squares)
sst = np.sum((data1['y'] - y_mean)**2)

# Calculate SSR (Residual Sum of Squares)
ssr = np.sum((data1['y'] - data1['predicted'])**2)

# Compute R-squared
r_squared_manual = 1 - (ssr / sst)
print(f"Manually Computed R-squared: {r_squared_manual:.3f}")

-----

❓ <span style="color:purple;">**Quick test**</span>:  **The fitted regression model has an R² value of 0.878. Which interpretation is correct?**

- **A.** The model predicts exactly 87.8% of the observations.
- **B.** The model explains approximately 87.8% of the variation in \(Y\).
- **C.** Approximately 87.8% of the residuals are equal to zero.
- **D.** Increasing \(X\) causes an 87.8% increase in \(Y\).

<details>
<summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;"><b>B.</b> The model explains approximately <b>87.8% of the variation in Y</b>.</p>

**Explanation**:

<p style="color:blue;">An R² value of 0.878 indicates that the regression model explains approximately <b>87.8% of the variation in the dependent variable Y</b>. The remaining 12.2% is not explained by the model and may be related to noise, omitted predictors, or model limitations.</p>

<p style="color:blue;"> When models are evaluated on the <b>same dataset</b>, a higher R² generally indicates a better fit.</p>

</details>

-----

#### We can also compare the manually computed R-Squared against the one returned by OLS. As shown below, they match each other.

In [ ]:
# R-squared obtained directly with OLS
r_squared = model.rsquared
print(f"R-squared: {r_squared:.3f}")

#### We can also use the root_mean_squared_error() method and r2_score() method from the scikit-learn library to compute the RMSE and the $R^2$

In [ ]:
from sklearn.metrics import root_mean_squared_error, r2_score

# Compute RMSE 
rmse = root_mean_squared_error(data1['y'], data1['predicted'])

# Compute R-squared
r_squared = r2_score(data1['y'], data1['predicted'])

# Print the results
print(f"RMSE: {rmse:.3f}")
print(f"R-squared: {r_squared:.3f}")

### As we can see, the output matches those calculated manually.

# Example 2: Non-linear regression

## <span style="color:blue"> Synthesize a data set which contains non-linearity and visualize it </span>

In [ ]:
n = 200
np.random.seed(42)
x = np.append(np.random.uniform(0, 4, n - 2), [2, 2.1])
y = 2.5 * x**3 + np.random.normal(0, 6, n)

# Create DataFrame
data2 = pd.DataFrame({"x": x, "y": y})

# Plot the data
plt.figure(figsize=(8, 6))
plt.scatter(data2['x'], data2['y'], alpha=0.6)
plt.title(f"Scatterplot of the dataset", fontsize=16)
plt.xlabel("x", fontsize=12)
plt.ylabel("y", fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

### As we can see in the above plot, there is a non-linear trend in the data.

## <span style="color:blue">Example 2.1: Fit the data set with a linear regression model, and print the formula corresponding to the fitted model

In [ ]:
# Apply OLS
X = sm.add_constant(data2['x'])  # Add constant term for the intercept
model = sm.OLS(data2['y'], X).fit()

# Extract coefficients
intercept = model.params['const']
slope = model.params['x']

# Print the formula
print(f"Fitted Regression Formula: y = {intercept:.3f} + {slope:.3f} * x")

data2['predicted'] = model.fittedvalues
data2['residuals'] = model.resid

### The linear regression model OLS fits to the data is Y = -29.08 + 35.004*X.

## <span style="color:blue">Example 2.2: Visualize the fitted linear regression model </span>

In [ ]:
plt.figure(figsize=(8, 6))

# Plot the data set
plt.scatter(data2['x'], data2['y'], alpha=0.6, label="Observed Data")

# Plot the fitted linear regression line
plt.plot(data2['x'], data2['predicted'], color='red', linewidth=2, label="Regression line")

# Titles and labels
plt.title("Dataset with Fitted Linear Regression Model", fontsize=16)
plt.xlabel("Independent Variable (X)", fontsize=12)
plt.ylabel("Dependent Variable (Y)", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

# Show the plot
plt.show()

### The blue points represent the observed data, which exhibits a non-linear trend. The red line is the fitted linear regression line.


-----

❓ <span style="color:purple;">**Quick test**</span>: **The fitted linear regression model does not appear to capture the curved relationship in the data. What would you expect to observe in its residual plot?**

- **A.** Residuals randomly scattered around zero with no visible pattern.
- **B.** A curved or U-shaped pattern in the residuals.
- **C.** All residuals equal to zero.
- **D.** Residuals forming a straight horizontal line above zero.

<details>
<summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;"><b>B.</b> We would expect a curved or U-shaped pattern in the residuals.</p>

**Explanation**:

<p style="color:blue;">The linear model appears to <b>underpredict</b> the observations at low and high values of \(X\), producing positive residuals. In the middle, it appears to <b>overpredict</b> the observations, producing negative residuals.</p>

<p style="color:blue;">Consequently, the residuals would likely form a <b>U-shaped pattern</b> rather than being randomly scattered around zero. This systematic pattern indicates that the linear model has not captured the nonlinear structure in the data.</p>

</details>

-----

## <span style="color:blue">Example 2.3:  Visualize the residuals in a scatter plot </span>

In [ ]:
plt.figure(figsize=(8, 6))

# Plot the residuals in a scatter plot
plt.scatter(data2['x'], data2['residuals'], alpha=0.6)

# Plot a horizontal red line for residuals = 0
plt.axhline(0, color='red', linestyle='--', linewidth=1)

# Titles and labels
plt.title("Residuals Plot", fontsize=16)
plt.xlabel("Independent Variable (X)", fontsize=12)
plt.ylabel("Residuals", fontsize=12)
plt.grid(True, alpha=0.3)

# Show the plot
plt.show()

### As we can see in the above scatter plot, the residuals are **not** randomly distributed around zero. Instead, it shows a clear pattern, indicating there are some non-linearity that are not captured by the linear regression model. This suggests **the linear model is not appropriate for the data, and a non-linear model would likely provide a better fit**. 


## <span style="color:blue"> Compute Root Mean Squared Error (RMSE) and the R-Sqaured ($R^2$)</span> <span style="color:red">

In [ ]:
from sklearn.metrics import root_mean_squared_error, r2_score

# Compute RMSE 
rmse = root_mean_squared_error(data2['y'], data2['predicted'])

# Compute R-squared
r_squared = r2_score(data2['y'], data2['predicted'])

# Print results
print(f"RMSE: {rmse:.3f}")
print(f"R-squared: {r_squared:.3f}")

-----

❓ <span style="color:purple;">**Quick test**</span>: **Based on the computed R² value, can we conclude that the linear model is actually a good fit?**

- **A. Yes**
- **B. No**

<details>
<summary>Click to reveal the answer</summary>

**Answer**: 

<p style="color:blue;"><b>B. No.</b></p>

**Explanation**:

<p style="color:blue;"> Although the fitted linear model has a relatively high $R^2$ of 0.836, the residual plot shows a clear curved pattern. This indicates that <b>the model captures the overall increasing trend but fails to capture the nonlinear structure in the data.</b> </p>
    
<p style="color:blue;"> Therefore, a high $R^2$ alone does not necessarily mean that the chosen model is appropriate. We should also <b>inspect the residuals and compare its performance with that of a nonlinear model.</b> </p>

</details>

-----

# Example 3: fit a polynomial regression model to the above non-linear data

### **Polynomial regression** is an extension of linear regression where the independent variable is raised to higher powers.

$$
\Large 
Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \dots + \beta_n X^n
$$

## <span style="color:blue"> Example 3.1: Fit a polynomial regression model to the data </span>

In [ ]:
# Set the degree of the polynomial
degree = 3 

# PolynomialFeatures() can be used to transforms a single input feature (x) 
# into polynomial features up to the specified degree
poly_features = PolynomialFeatures(degree=degree)
# Transform 'x' to polynomial features
X_poly = poly_features.fit_transform(data2[['x']])  

pd.DataFrame(
    X_poly,
    columns=poly_features.get_feature_names_out(["x"])
).head()

In [ ]:
# Fit a linear regression model based on the transformed polynomial feature
model = LinearRegression()
model.fit(X_poly, data2['y'])

# Store the predicted values
data2['predicted'] = model.predict(X_poly)

# Calculate residuals: observed - predicted
data2['residuals'] = data2['y'] - data2['predicted']

# Extract the intercept and coefficients
intercept = model.intercept_
coefficients = model.coef_

### In the above code, the `LinearRegression()` method from scikit-learn library is used to find a polynomial regression model to the data. 

### First of all, <small>`PolynomialFeatures(degree=degree)`</small> transforms a single input feature (x) into polynomial features up to the specified degree. For example, if degree=3, it generates $[1, x, x^2, x^3]$, where 1 is the intercept term, $x$ is the original feature, $x^2, x^3$ are the additional polynomial terms. 

### The resulting <small>`X_poly`</small> is a 2D array where each row contains $[1, x, x^2, x^3]$.

### Next, we fit a linear regression model using the polynomial features (<small>`X_poly`</small>) as input and the target variable (`y`) as output.

### The intercept and coefficients ($\beta_0, \beta_1, \beta_2, \beta_3, \dots$) can be extracted from <small>`intercept_`</small> and <small>`coef_`</small> of the fitted model.

## <span style="color:blue">Example 3.2: Display the intercept and coefficients of the fitted polynomial regression model  </span>

In [ ]:
# Create a list to store the individual term of the polynomial regression model
terms = ['Intercept'] + [f"x^{i}" for i in range(1, degree + 1)]

# Create a list which stores the intercept and individual coefficient values
# The coefficient for x^0 is excluded as it's already included in the intercept
values = [intercept] + list(coefficients[1:]) 

# Create a data frame which contains the term and the coefficients for 
# better visualization
coef_df = pd.DataFrame({"Term": terms, "Coefficient": values})
coef_df

### In the above code, a data frame is created to store the individual term of the fitted polynomial regression model, as well as the intercept and coefficients values corresponding to each term. At the end, the data frame is displayed.

## <span style="color:blue">Example 3.3: Build the formula corresponding to the fitted polynomial regression model, based on the intercept and coefficients </span>

In [ ]:
# Start the formula with the intercept
formula = f"Y = {intercept:.2f}" 

# Extend the formula with the coefficient multiplied with the corresponding term
for i in range(1, degree + 1):
    formula += f" + {coefficients[i]:.2f} * X^{i}"

# Print the formula
print("Fitted polynomial regression model is:", formula)

### As we can see, the formula corresponding to the fitted polynomial regression model is $Y = 0.58 + 0.09*X + 0.18*X^2 + 2.41*X^3$

## <span style="color:blue">Example 3.4: Plot the data and the fitted polynomial regression model </span>

In [ ]:
plt.figure(figsize=(10, 6))

# Plot a scatter plot for the data
plt.scatter(data2['x'], data2['y'], alpha=0.6, label="Observed Data")

# Sort the data by 'x' for smooth plotting of the regression line
sorted_data = data2.sort_values(by='x')

# Plot the polynomial regression line
plt.plot(sorted_data['x'], sorted_data['predicted'], color='red', label='Polynomial regression line')

# Titles and labels
plt.title("Polynomial Regression Model", fontsize=16)
plt.xlabel("Independent Variable (X)", fontsize=12)
plt.ylabel("Dependent Variable (Y)", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

# Show the plot
plt.show()

### According to the above plot, the fitted polynomial regression model seems capture well the pattern in the data. Next, we can examine the residual plot to further confirm whether it is a good fit.

## <span style="color:blue">Example 3.5: Visualize and examine the residuals in a scatter plot </span>

In [ ]:
plt.figure(figsize=(8, 6))

# Plot the residuals in a scatter plot
plt.scatter(data2['x'], data2['residuals'], alpha=0.6)

# Plot a horizontal red line for residuals = 0
plt.axhline(0, color='red', linestyle='--', linewidth=1)

# Titles and labels
plt.title("Residuals Plot", fontsize=16)
plt.xlabel("Independent Variable (X)", fontsize=12)
plt.ylabel("Residuals", fontsize=12)
plt.grid(True, alpha=0.3)

# Show the plot
plt.show()

## <span style="color:blue">Example 3.6: Compute Root Mean Squared Error (RMSE) and the R-Sqaured ($R^2$) </span>

In [ ]:
from sklearn.metrics import root_mean_squared_error, r2_score

# Compute RMSE 
rmse = root_mean_squared_error(data2['y'], data2['predicted'])

# Compute R-squared
r_squared = r2_score(data2['y'], data2['predicted'])

# Print the results
print(f"RMSE: {rmse:.3f}")
print(f"R-squared: {r_squared:.3f}")

-----

❓ <span style="color:purple;">**Quick test**</span>:  **Based on the residual plot, RMSE, and R², can we conclude that the polynomial regression model fits the observed data better than the linear regression model?**

- **A. Yes**
- **B. No**

<details>
<summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;"><b>A. Yes.</b></p>

**Explanation**:

<ul style="color:blue; font-size:16px; font-family:inherit; line-height:1.6;">
  <li>The polynomial model’s residuals are more randomly scattered around zero and no longer show the clear curved pattern produced by the linear model.</li>
  <li>Its <b>RMSE decreases from 18.11 to 5.737</b>, indicating smaller prediction errors on the observed data.</li>
  <li>Its <b>R² increases from 0.836 to 0.984</b>, indicating that it explains a larger proportion of the variation in Y.</li>
  <li>Therefore, the polynomial model provides a better <b>in-sample fit</b>. However, its performance on new observations should still be evaluated using unseen test data.</li>
</ul>

</details>

-----

#  Example 4: Multiple Linear Regression 

### The multiple linear regression model extends the simple linear regression to multiple independent variables $X_1$, $X_2$, $\dots$, $X_n$, 

$$
\Large 
Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_n X_n
$$

### In this example, we try to find a multiple linear regression model for the house price data set. We first load the data set.

In [ ]:
house = pd.read_csv('house_sales.csv', sep='\t')
house.head(10)

### For illustration purpose, we will make use of  Total Living Square Feet (**SqFtTotLiving**), Year Built (**YrBuilt**), and the Land Value (**LandVal**) as the independent variables. The dependent variable is the Sale Price (**SalePrice**).

In [ ]:
subset = ['SqFtTotLiving', 'YrBuilt', 'LandVal','SalePrice']
house[subset].head(10)

### In the above code, we extract and display the independent and dependent variables relevant for multiple linear regression.

## <span style="color:blue"> Example 4.1: fit a multiple linear regression model to the data

In [ ]:
predictors = ['SqFtTotLiving', 'YrBuilt',  'LandVal']
outcome = 'SalePrice'

house_lm = LinearRegression()
house_lm.fit(house[predictors], house[outcome])

# Create a data frame to store the intercept and coefficients
coefficients_df = pd.DataFrame({
    'Term': ['Intercept'] + predictors,
    'Coefficient': [house_lm.intercept_] + list(house_lm.coef_)
})

coefficients_df

### In the above code, scikit-learn’s LinearRegression() method is used for multiple linear regression. 

### The intercept and the coefficients for each independent variable can be retrieved from the <small>`intercept_`</small> and <small>`coef_`</small> of the fitted model

### At the end, the intercept and coefficients are stored in a data frame for better visualization.

## <span style="color:blue"> Example 4.2: derive formula corresponding to the fitted multiple linear regression model

In [ ]:
# Derive the formula corresponding to the fitted model
formula = f"{outcome} = {house_lm.intercept_:.2f}"

for name, coef in zip(predictors, house_lm.coef_):
    formula += f" + ({coef:.2f}) * {name}"

print("Fitted multiple linear regression model is: ", formula)

The above code derives the formula corresponding to the fitted multiple linear regression model, which is  **SalePrice = -690331.80 + (147.27) * SqFtTotLiving + (319.81) * YrBuilt + (1.19) * LandVal**

The intercept (-690331.80) is the predicted value of the SalePrice when all independent variables are zero. While this value may not have practical meaning (since a house cannot have 0 square feet, be built in year 0, or have a land value of 0), it serves as a baseline or starting point for the model's predictions.

The coefficient for the SqFtTotLiving is 147.27. It means **for every additional square foot of total living area, the SalePrice increases by approximately 147.27**, assuming YrBuilt and LandVal remain constant. 

Similar reasoning can be applied to the coefficient for the other two independent variables. 

# 🏠 Assignment Questions <span style="color:blue;"> [20/20 points] </span>

## <span style="color:blue;"> [5 points] </span> Question 1: investigate the impact of outliers on linear regression

### <span style="color:blue"> Synthesize the dataset to be used for this question </span>

In [ ]:
# Create a data set with 60 data points
n = 60
np.random.seed(3)
x = np.append(np.random.uniform(0, 4, n - 2), [2, 2.1])
y = 5.6 * x + np.random.normal(0, 2, n)

# Create a  data frame based on the X and y value
data3 = pd.DataFrame({"x": x, "y": y})

# Plot the data set
plt.figure(figsize=(8, 6))
plt.scatter(data3['x'], data3['y'], alpha=0.6)
plt.title(f"Scatterplot of the original dataset", fontsize=16)
plt.xlabel("x", fontsize=12)
plt.ylabel("y", fontsize=12)
plt.ylim(0, 65)
plt.grid(True, alpha=0.3)
plt.show()

The above scatterplot shows a **clear linear relationship** between the independent variable $X$ and the dependent variable $Y$. And there are **no obvious outliers**.

### <span style="color:blue"> Replace 8 data points in the original data set with extreme outliers, and build a new data set </span>

In [ ]:
np.random.seed(42)  # random seed for reproducibility

# Create a copy of the original data set 
data3_with_outliers = data3.copy()

# Randomly pick 8 indices, later we will change their y-values
outlier_indices = np.random.choice(data3_with_outliers.index, # pick from existing indices
                                   size=8,                    # select exactly 8 data points
                                   replace=False              # ensure the same data point will not be selected twice
                                  )

# Increase the y-value of the selected data points by 40. 
# This make these points extreme in their y-value 
data3_with_outliers.loc[outlier_indices, 'y'] += 40

print("Selected outlier indices:", outlier_indices.tolist())
print("\nOriginal data of the selected data points: ")
display(data3.loc[outlier_indices])

print("Outlier data points (y-value increased by 40):")
display(data3_with_outliers.loc[outlier_indices])

The above code randomly selects 8 data points from the original data set, and **increase their y-value by 40**. **This will make these data points appear as outliers**. And we obtained a new dataset with outliers, called `data3_with_outliers`.

### <span style="color:blue"> Visualize the original data set and the added outliers in a scatterplot </span>

In [ ]:
plt.figure(figsize=(8, 6))

# Plot the original data set
plt.scatter(data3['x'], data3['y'], alpha=0.6, label='Original dataset')

# Highlight outliers
plt.scatter(
    data3_with_outliers.loc[outlier_indices, 'x'],
    data3_with_outliers.loc[outlier_indices, 'y'],
    edgecolor='black', s=80, label='Added outliers'
)

plt.title("Oringial dataset with the added outliers")
plt.xlabel("x")
plt.ylabel("y")
plt.ylim(0, 65)
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

### <span style="color:blue"> Q1.1: Fit a linear regression model (using OLS) to the original data set (`data3`), and print out the formula corresponding to the fitted model; the RMSE value and the R-squared value.

In [ ]:
# insert your code here

### <span style="color:blue"> Q1.2: Fit a linear regression model (using OLS) to the data set that contains outliers (`data3_with_outliers`), and print the formula corresponding to the fitted model, the RMSE value and the R-squared value

In [ ]:
# insert your code here

### <span style="color:blue"> Q1.3: Visualize the original dataset, the added outliers, the fitted linear regression line for the original dataset, and the fitted linear regression line for the dataset with outliers in a single plot </span>

In [ ]:
# insert your code here

### <span style="color:blue"> Q1.4: Based on the output plot of Q1.3, what do you observe? Based on the observation, what conclusion can we draw? Explain your answer. </span>

----
<div>
    
**Answer**: insert your answer here


</div>

----

## <span style="color:blue;"> [5 points] </span> Question 2: synthesis a dataset, fit a linear regression model to it, and assess whether the model provides a good fit.

### <span style="color:blue"> Generate a synthesized data set for Q2 and Q3 </span>

In [ ]:
n = 200
np.random.seed(42)
x = np.append(np.random.uniform(0, 4, n - 2), [2, 2.1])
# Synthesize a sine curve with noise
y = 25 * np.sin(2 * np.pi * x / 4) + np.random.normal(0, 5, n)

# Create DataFrame
data4 = pd.DataFrame({"x": x, "y": y})

# Plot the data
plt.figure(figsize=(8, 6))
plt.scatter(data4['x'], data4['y'], alpha=0.6)
plt.title(f"Scatterplot of the dataset", fontsize=16)
plt.xlabel("x", fontsize=12)
plt.ylabel("y", fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

### <span style="color:blue"> Q2.1: Fit a linear regression model to the data using OLS, and print the formula corresponding to the fitted model

In [ ]:
### insert your code here

### <span style="color:blue"> Based on the output of the above code, answer what is the formula corresponding to the regression model </span>

----
<div>
    
**Answer**: insert your answer here


</div>

----

### <span style="color:blue"> Q2.2: Visualize the linear regression model </span>

In [ ]:
### insert your code here

### <span style="color:blue"> Q2.3: Visualize the residuals in a scatter plot </span>

In [ ]:
### insert your code here

### <span style="color:blue"> Q2.4: Based on the residual plot, answer is linear regression model appropriate for the data? Explain your answer. </span>

----
<div>
    
**Answer**: insert your answer here


</div>

----

### <span style="color:blue"> Q2.5: Compute and print Root Mean Squared Error (RMSE) and the R-Sqaured ($R^2$) </span>

In [ ]:
### insert your code here

## <span style="color:blue;"> [10 points] </span> Question 3: fit a polynomial regression model to the same dataset and assess whether a polynomial regression model provides a good fit.

We will make a copy of dataset `data4` before fitting a new model to the same dataset. **Please use `data5` for all the subquestions in this exercise**.

In [ ]:
data5 = data4.copy()

### <span style="color:blue"> Q3.1: Fit a polynomial regression model to the data, with the degree up to 3 </span>

In [ ]:
### insert your code here

### <span style="color:blue"> Q3.2: Display the intercept and coefficients of the fitted polynomial regression model  </span>

In [ ]:
### insert your code here

### <span style="color:blue"> Q3.3: Build the formula corresponding to the fitted polynomial regression model, based on the intercept and coefficients </span>

In [ ]:
### insert your code here

### <span style="color:blue"> Based on the output of the above code, answer what is the formula corresponding to the regression model </span>

----
<div>
    
**Answer**: insert your answer here


</div>

----

### <span style="color:blue"> Q3.4: Plot the data and the fitted polynomial regression model </span>

In [ ]:
# insert your code here

### <span style="color:blue"> Q3.5: Visualize the residuals in a scatter plot </span>

In [ ]:
# insert your code here

### <span style="color:blue"> Q3.6: Based on the residual plot, answer is the polynomial regression model a good fit to the data? Explain your answer. </span>

----
<div>
    
**Answer**: insert your answer here


</div>

----

### <span style="color:blue"> Q3.7: Compute and print Root Mean Squared Error (RMSE) and the R-Sqaured ($R^2$) </span>

In [ ]:
# insert your code here

### <span style="color:blue"> Q3.8: Display the Root Mean Squared Error (RMSE) and the R-Sqaured ($R^2$) computed using the linear regression model and the polynomial regression model side-by-side in a single table. </span>

In [ ]:
# insert your code here

### <span style="color:blue"> Q3.9: based on the output in Q3.8, answer how can we use the RMSE and R-Squared values to compare the two models fitted to the same dataset. Explain your answer. </span>

----
<div>
    
**Answer**: insert your answer here


</div>

----